# D212 - Task 3: Market Basket Analysis
Desiree McElroy | WGU D212 Data Mining II

## Part I: Research Question

**A1 — Research Question:** What prescription combinations are most frequently co-prescribed together among patients in this dataset, and can those associations inform pharmacists and medical care teams about predictable prescription patterns?

**A2 — Goal:** Identify top prescription item associations using the Apriori algorithm to reveal co-prescription patterns that may support clinical decision making and pharmacy planning. *(See written report for full narrative.)*

## Part II: Market Basket Justification *(See written report for full explanation.)*

**B1 — Explanation:** Market basket analysis treats each patient row as a transaction. The Apriori algorithm identifies drug combinations co-occurring above a minimum support threshold and scores each rule by support, confidence, and lift. 

**B2 — Example transaction:** Patient 1 (row 2 of raw data): amlodipine, albuterol aerosol, allopurinol, pantoprazole, lorazepam, omeprazole, mometasone, fluconazole, gabapentin, pravastatin, losartan, metoprolol succinate XL, sulfamethoxazole, spironolactone, albuterol HFA, levofloxacin, promethazine, glipizide (20 prescriptions).

**B3 — Assumption:** Item order within a transaction is irrelevant (bag-of-items). Each transaction is treated independently with no awareness of shared providers or diagnoses.

# Part III: Data Preparation and Analysis

C.  Prepare and perform market basket analysis by doing the following:

In [1]:
## imports
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_rows', None)

#### Data Acquire & Explore

In [2]:
df = pd.read_csv('medical_market_basket.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15002 entries, 0 to 15001
Data columns (total 20 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Presc01  7501 non-null   object
 1   Presc02  5747 non-null   object
 2   Presc03  4389 non-null   object
 3   Presc04  3345 non-null   object
 4   Presc05  2529 non-null   object
 5   Presc06  1864 non-null   object
 6   Presc07  1369 non-null   object
 7   Presc08  981 non-null    object
 8   Presc09  654 non-null    object
 9   Presc10  395 non-null    object
 10  Presc11  256 non-null    object
 11  Presc12  154 non-null    object
 12  Presc13  87 non-null     object
 13  Presc14  47 non-null     object
 14  Presc15  25 non-null     object
 15  Presc16  8 non-null      object
 16  Presc17  4 non-null      object
 17  Presc18  4 non-null      object
 18  Presc19  3 non-null      object
 19  Presc20  1 non-null      object
dtypes: object(20)
memory usage: 2.3+ MB


In [3]:
## get a view of dataframe
df.head(10)

,Presc01,Presc02,Presc03,Presc04,Presc05,Presc06,Presc07,Presc08,Presc09,Presc10,Presc11,Presc12,Presc13,Presc14,Presc15,Presc16,Presc17,Presc18,Presc19,Presc20
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,amlodipine,albuterol aerosol,allopurinol,pantoprazole,lorazepam,omeprazole,mometasone,fluconozole,gabapentin,pravastatin,cialis,losartan,metoprolol succinate XL,sulfamethoxazole,abilify,spironolactone,albuterol HFA,levofloxacin,promethazine,glipizide
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,citalopram,benicar,amphetamine salt combo xr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,enalapril,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,paroxetine,allopurinol,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,abilify,atorvastatin,folic acid,naproxen,losartan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Quite a few rows with many null values. This is to be expected. Definitely worth checking if some rows have **all null values**, in which case they should be removed.

In [4]:
# how many rows are completely empty?
print(f"Total rows: {df.shape[0]}")

print(f"Rows with all null values: {df.isnull().all(axis=1).sum()}")

print(f"Rows remaining after dropping: {df.notna().any(axis=1).sum()}")
print('-------------')
print(f'Displaying Dataframe with NaN across board.')
df[df.isnull().all(axis=1)].head()

Total rows: 15002
Rows with all null values: 7501
Rows remaining after dropping: 7501
-------------
Displaying Dataframe with NaN across board.


,Presc01,Presc02,Presc03,Presc04,Presc05,Presc06,Presc07,Presc08,Presc09,Presc10,Presc11,Presc12,Presc13,Presc14,Presc15,Presc16,Presc17,Presc18,Presc19,Presc20
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Below, printing a view of the way the apriori algorithm wants the data preprocessed.

In [5]:
## viewing prep for algorithm
df.apply(lambda row: row.dropna().tolist(), axis=1).tolist()[:2]

[[],
 ['amlodipine',
  'albuterol aerosol',
  'allopurinol',
  'pantoprazole',
  'lorazepam',
  'omeprazole',
  'mometasone',
  'fluconozole',
  'gabapentin',
  'pravastatin',
  'cialis',
  'losartan',
  'metoprolol succinate XL',
  'sulfamethoxazole',
  'abilify',
  'spironolactone',
  'albuterol HFA',
  'levofloxacin',
  'promethazine',
  'glipizide']]

Below, I wanted to look at is the individual prescription names and sort them. This data set may be clean, but as due diligence, it is best to view all of these names in order to identify any possible misspellings.

In [6]:
## wanting to look at individual names to identify any possible misspellings
all_items = pd.Series(df.values.flatten()).dropna()
item_counts = all_items.value_counts().sort_index()
print(f"Total unique prescriptions: {len(item_counts)}")
print('Prescription Names')
item_counts.sort_index(ascending=True)

Total unique prescriptions: 119
Prescription Names


Duloxetine                      90
Premarin                       351
Yaz                             32
abilify                       1788
acetaminophen                  118
actonel                         90
albuterol HFA                   67
albuterol aerosol              153
alendronate                     36
allopurinol                    250
alprazolam                     595
amitriptyline                   34
amlodipine                     536
amoxicillin                     65
amphetamine                    226
amphetamine salt combo         513
amphetamine salt combo xr     1348
atenolol                        81
atorvastatin                   972
azithromycin                   107
benazepril                      69
benicar                        157
boniva                          46
bupropion sr                    31
carisoprodol                    86
carvedilol                    1306
cefdinir                        14
celebrex                        82
celecoxib           

### C2 — Using Apriori Algorithm

#### Data Preprocess

1. Lower case all entries.
1. Remove rows where all entries are null.
1. Create "list of list" as preparation for apriori algorithm (referenced [transform encoder here](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/#example-1-generating-frequent-itemsets)).
1. From "list of list", transform to boolean array using same encoder.
1. Create view of itemsets from apriori algorithm.
1. Create association rules table for analysis and exporting.


In [7]:
## make all entries lower case to avoid issues with case sensitivity when viewing
df = df.apply(lambda col: col.str.lower())
## remove rows where all entries are null
cleaned_df = df[df.notna().any(axis=1)]

In [8]:
## adding arg for re-running notebook without overwriting csv
save_csv = True
if save_csv:
    cleaned_df.to_csv('cleaned_df_basket_analysis.csv', index=False)

In [9]:
# convert each row to a list of prescription values, creating a list of list for transaction encoder
transactions = cleaned_df.apply(lambda row: row.dropna().tolist(), axis=1).tolist()

# wanting to spot check
print(f"Number of transactions: {len(transactions)}")
print(f"Sample transaction: {transactions[:5]}")

Number of transactions: 7501
Sample transaction: [['amlodipine', 'albuterol aerosol', 'allopurinol', 'pantoprazole', 'lorazepam', 'omeprazole', 'mometasone', 'fluconozole', 'gabapentin', 'pravastatin', 'cialis', 'losartan', 'metoprolol succinate xl', 'sulfamethoxazole', 'abilify', 'spironolactone', 'albuterol hfa', 'levofloxacin', 'promethazine', 'glipizide'], ['citalopram', 'benicar', 'amphetamine salt combo xr'], ['enalapril'], ['paroxetine', 'allopurinol'], ['abilify', 'atorvastatin', 'folic acid', 'naproxen', 'losartan']]


In [10]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
te_array[0]

array([ True, False, False,  True,  True, False,  True, False, False,
        True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False,  True, False,  True, False, False,
       False, False, False, False, False, False,  True, False, False,
        True,  True, False, False, False, False, False, False,  True,
       False,  True, False,  True, False,  True, False, False, False,
        True, False, False, False,  True, False, False, False, False,
       False, False,  True,  True, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False])

In [11]:
te_df = pd.DataFrame(data=te_array, columns=te.columns_ )
te_df.head()

,abilify,acetaminophen,actonel,albuterol aerosol,albuterol hfa,alendronate,allopurinol,alprazolam,amitriptyline,amlodipine,...,triamcinolone ace topical,triamterene,trimethoprim ds,valaciclovir,valsartan,venlafaxine xr,verapamil sr,viagra,yaz,zolpidem
0,True,False,False,True,True,False,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [12]:
# Find frequent itemsets - min_support is the minimum frequency threshold
frequent_itemsets = apriori(te_df, min_support=0.01, use_colnames=True)
frequent_itemsets.sort_values('support', ascending=False).head(15)

,support,itemsets
0,0.238368,(abilify)
9,0.179709,(amphetamine salt combo xr)
15,0.174110,(carvedilol)
38,0.170911,(glyburide)
25,0.163845,(diazepam)
47,0.132116,(losartan)
11,0.129583,(atorvastatin)
46,0.098254,(lisinopril)
52,0.095321,(metoprolol)
27,0.095054,(doxycycline hyclate)


**Takeaway**

23.8% of the patient transactions in the dataset include abilify, roughly 1,785 out of 7,501 patients. 
That's a fairly high single-item support, which means abilify will likely show up in a lot of association rules.

### C3 — Association Rules Table

In [13]:
# Generate rules - min_threshold applies to confidence
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(['lift','confidence'], ascending=False)

rules.head(15)

,antecedents,consequents,support,confidence,lift
299,(methylprednisone),(lisinopril),0.015998,0.323450,3.291994
298,(lisinopril),(methylprednisone),0.015998,0.162822,3.291994
377,"(abilify, carvedilol)",(lisinopril),0.017064,0.285714,2.907928
380,(lisinopril),"(abilify, carvedilol)",0.017064,0.173677,2.907928
367,"(abilify, carvedilol)",(glipizide),0.010265,0.171875,2.609786
370,(glipizide),"(abilify, carvedilol)",0.010265,0.155870,2.609786
90,(amphetamine salt combo),(metoprolol),0.016131,0.235867,2.474464
91,(metoprolol),(amphetamine salt combo),0.016131,0.169231,2.474464
77,(amlodipine),(metoprolol),0.016664,0.233209,2.446574
76,(metoprolol),(amlodipine),0.016664,0.174825,2.446574


### C4 — Top Three Rules

Top rules by lift (see written report for full analysis):
1. methylprednisone → lisinopril `(lift 3.292)`
2. {abilify, carvedilol} → lisinopril `(lift 2.908)`
3. {abilify, carvedilol} → glipizide `(lift 2.610)`

## Part IV: Data Summary and Implications *(See written report for full analysis.)*

**D1 — Significance of support, lift, and confidence:** Support ranged 0.010–0.039; lift ranged 1.0–3.29; top rule confidence ranged 0.16–0.42. 

**D2 — Practical significance:** Top rules point to a patient segment managing cardiovascular and mental health comorbidities concurrently.

**D3 — Recommendation:** Implement co-prescription awareness flags for methylprednisone/lisinopril and abilify combinations in pharmacy management systems.